# Bi-encoder — tầng truy hồi dày (dense)**Mục đích:** nhóm đang thiếu hẳn nửa dày của kiến trúc chuẩn (bi-encoder + cross-encoder).7 câu dev300 có gold NGOÀI top-100 BM25 = **2,17 điểm** E hiện không chạm tới.**Ngưỡng đặt TRƯỚC khi chạy:**| đo được | quyết định ||---|---|| recall@100 hybrid **> 0.9783** | rổ tốt hơn → dùng, 2,17 điểm mở ra || recall@5 riêng dense **> 0.80** | tín hiệu mạnh (BM25 thô = 0.7533) → hoà vào điểm cuối || recall@5 riêng dense **< 0.70** | yếu → chỉ dùng làm tín hiệu phụ |**Upload lên dataset `project-ir`:** `bi_encoder.py` (MỚI). Mọi thứ khác đã có sẵn.GPU T4 · Internet **On** (cần tải model) · Save & Run All.

In [ ]:
!pip -q install sentence-transformers 2>&1 | tail -1

In [ ]:
# ===== Bước 0: đường dẫn + dấu vân tay =====import os, sys, json, hashlib, timeimport numpy as npINPUT_DIR = next(p for p in ("/kaggle/input/project-ir",                             "/kaggle/input/datasets/locdovan211/project-ir")                 if os.path.isdir(p))CTX = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",                       f"{INPUT_DIR}/selected-contexts")           if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))OUT = "/kaggle/working"sys.path.insert(0, INPUT_DIR)for f in ("bi_encoder.py", "make_candidates_fallback.py"):    p = f"{INPUT_DIR}/{f}"    b = open(p, "rb").read()    print(f"{f:32} {len(b):>7} bytes  {hashlib.sha256(b).hexdigest()[:12]}")import bi_encoder as BEBE.selftest()                      # tái lập số đã biết TRƯỚC khi lên GPUprint("INPUT_DIR =", INPUT_DIR)print("CTX       =", CTX, "·", len(os.listdir(CTX)), "file")

In [ ]:
# ===== Bước 1: đếm mảnh trên CPU, ước chi phí (quy tắc 6) =====t0 = time.time()n_chunk = BE.count(CTX)print(f"đếm mất {time.time()-t0:.0f}s")assert n_chunk < 400_000, "nhiều mảnh bất thường — kiểm EMBED_CHARS trước khi đốt GPU"

In [ ]:
# ===== Bước 2: nhúng cả kho. Lưu theo lô -> đứt vẫn chạy tiếp =====from sentence_transformers import SentenceTransformermodel = SentenceTransformer(BE.MODEL_NAME, device="cuda")model.max_seq_length = 2048t0 = time.time()meta_p = BE.encode_corpus(model, CTX, OUT)print(f"xong {time.time()-t0:.0f}s -> {meta_p}")

In [ ]:
# ===== Bước 3: dense chạy MỘT MÌNH tốt tới đâu? =====V, owner = BE.load(OUT)print("ma trận:", V.shape, "·", len(set(owner)), "văn bản")dev  = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))gold = {q: {str(x) for x in v["answer"]} for q, v in dev.items()}qv = model.encode([dev[q]["question"] for q in dev], batch_size=32,                  normalize_embeddings=True).astype(np.float32)dense_rank = {}for i, q in enumerate(dev):    s = BE.doc_scores(qv[i], V, owner)    dense_rank[q] = sorted(s, key=lambda d: -s[d])[:100]json.dump(dense_rank, open(f"{OUT}/dense_top100_dev300.json", "w"), ensure_ascii=False)rec = lambda R, k: sum(len(gold[q] & set(R[q][:k])) / len(gold[q]) for q in gold) / len(gold)print("\n== DENSE MỘT MÌNH ==")for k in (1, 5, 10, 25, 50, 100):    print(f"  recall@{k:<4} = {rec(dense_rank,k):.4f}")r5 = rec(dense_rank, 5)print(f"\n>> recall@5 = {r5:.4f}  (BM25 thô = 0.7533)")print("   " + ("MẠNH, hoà vào điểm cuối" if r5 > .80 else               "YẾU, chỉ dùng phụ" if r5 < .70 else "trung bình — xem hybrid"))

In [ ]:
# ===== Bước 4: hybrid — rổ ứng viên có tốt lên không? =====S = json.load(open(f"{INPUT_DIR}/scores_dev300_bm25pick_merge1800_M20_K20.json", encoding="utf-8"))bm = {q: sorted(S[q], key=lambda d: -S[q][d]["bm25"]) for q in S}print("recall@100 BM25 hiện tại :", f"{rec(bm,100):.4f}   <- mốc phải vượt")for k in (25, 50, 100):    hy = {q: list(dict.fromkeys(bm[q] + dense_rank[q][:k])) for q in gold}    print(f"  + dense top-{k:<4} -> recall@100+ = {rec(hy, 1000):.4f}   "          f"(rổ trung bình {sum(len(v) for v in hy.values())/len(hy):.0f} văn bản)")

In [ ]:
# ===== Bước 5: dense làm TÍN HIỆU THỨ BA cho recall@5 (CPU, 0 giây GPU) =====ce = lambda v: max(v.get("ce", -9), v.get("ce_deep", -9))dsc = {q: BE.doc_scores(qv[i], V, owner, docs=set(S[q])) for i, q in enumerate(dev)}def top5(q, w, n=2):    e = S[q]    out = list(dict.fromkeys(sorted(e, key=lambda d: -e[d]["bm25"])[:n]))    for d in sorted(e, key=lambda d: -(ce(e[d]) + w * dsc[q].get(d, 0))):        if len(out) >= 5: break        if d not in out: out.append(d)    return out[:5]print("trọng số dense  recall@5")base = Nonefor w in (0, .05, .1, .2, .3, .5, 1.0):    r = sum(len(gold[q] & set(top5(q, w))) / len(gold[q]) for q in gold) / len(gold)    base = base if base is not None else r    print(f"  w={w:<5} {r:.4f}   {100*(r-base):+.2f}")print("\nMốc hiện tại 0.9183. Quy tắc 4: dưới +2,0 thì ĐỪNG mang lên đề thi.")

In [ ]:
# ===== Bước 6: chốt tài sản — TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN (quy tắc 2) =====import globprint("\n".join(f"{os.path.basename(p):28} {os.path.getsize(p)/1e6:7.1f} MB"                for p in sorted(glob.glob(f"{OUT}/*"))))print("\nBẮT BUỘC tải: emb_*.npy + emb_meta.json  -> có chúng thì mọi thí nghiệm sau chạy CPU.")